# 🕸️ 04 — Drishti: Cascade Simulator (Network Propagation)

**This is our INNOVATION differentiator.** Every other team will build a single-train predictor. We model the Indian Railways network as a graph and compute **how a delay on one train propagates to downstream trains sharing the same stations**.

Approach: We use plain PySpark joins (not GraphFrames, which is flaky on Free Edition) to simulate 2-hop cascade propagation. Conceptually identical, more reliable.

In [0]:
from pyspark.sql import functions as F

# Build a station→train co-occurrence table: which trains share which station
co_occur = spark.table("setu_rail.gold.features_delay_ml") \
    .select("train_no", "station_code", "scheduled_hour") \
    .distinct()

co_occur.write.format("delta").mode("overwrite") \
    .saveAsTable("setu_rail.gold.station_train_graph")

print(f"✅ station_train_graph: {co_occur.count():,} edges")

In [0]:
def simulate_cascade(source_train: str, source_station: str, delay_min: float,
                     decay: float = 0.6, hops: int = 2) -> list:
    """
    Simulate cascade: find trains that share the source station and
    arrive within 2 hours of the source, then propagate delay with decay.

    Returns a list of dicts:
      [{train_no, station, hop, propagated_delay}, ...]
    """
    g = spark.table("setu_rail.gold.station_train_graph")

    # Find source train's hour at source station
    src_row = g.filter((F.col("train_no") == source_train) &
                       (F.col("station_code") == source_station)) \
               .select("scheduled_hour").limit(1).collect()
    if not src_row:
        return []
    src_hour = src_row[0]["scheduled_hour"]

    # Hop 1: same station, within 2 hours
    hop1 = g.filter((F.col("station_code") == source_station) &
                    (F.col("train_no") != source_train) &
                    (F.abs(F.col("scheduled_hour") - F.lit(src_hour)) <= 2)) \
            .limit(10)

    affected = [
        {
            "train_no":  r.train_no,
            "station":   r.station_code,
            "hop":       1,
            "propagated_delay": round(delay_min * decay, 1),
        }
        for r in hop1.collect()
    ]

    # Hop 2: follow affected trains to their downstream stations
    if hops >= 2 and affected:
        hop1_trains = [a["train_no"] for a in affected]
        hop2 = g.filter(F.col("train_no").isin(hop1_trains) &
                        (F.col("station_code") != source_station)) \
                .limit(15)
        for r in hop2.collect():
            affected.append({
                "train_no":  r.train_no,
                "station":   r.station_code,
                "hop":       2,
                "propagated_delay": round(delay_min * (decay ** 2), 1),
            })

    return affected

In [0]:
# Demo: simulate a major delay on train 12000 at Chennai Central
result = simulate_cascade("12000", "MAS", 60.0)
print(f"Cascade affected {len(result)} trains:")
for r in result[:15]:
    print(f"  Hop {r['hop']}: Train {r['train_no']:>6} @ {r['station']:<6} → +{r['propagated_delay']:>5} min")

In [0]:
# Save the function as a registered UDF for SQL access — judges can call it from Genie
spark.udf.register(
    "simulate_cascade_udf",
    lambda t, s, d: str(simulate_cascade(t, s, float(d))),
    "string",
)
print("✅ Cascade UDF registered")

✅ **Next:** `03_vani_rag_pipeline/01_chunk_and_embed_rules.ipynb`